<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_11_practicum_search/note_lesson_11_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 11 — Практикум П2. Пошук

> Другий практикум алгоритмічної трійки М1 (П1 → П2 → П3):
>
> ```
>               ALGORITHMIC THINKING
>                      │
>           ┌──────────┼──────────┐
>           ▼          ▼          ▼
>        П1            П2          П3
>     EVALUATE        CHOOSE     REPRESENT
>    complexity      strategy    data structure
>        │              │            │
>        └──────────────┼────────────┘
>                       ▼
>              TRADE-OFF THINKING
> ```
>
> П1 навчив оцінювати ціну рішення. Тут — про вибір: **«Що я знаю про структуру даних, і як це дозволяє мені не перевіряти все?»** 2 години, чотири стратегії пошуку, нерівна глибина: лінійний пошук — коротко (ви його вже фактично вмієте), бінарний пошук — короткий приклад, два вказівники і sliding window — основна практика.

Структура: **RETRIEVE → CONCEPT → PREDICT → RUN → INVESTIGATE → CREATE → TRANSFER**.

## 🔁 RETRIEVE — пригадай попередні уроки (без підглядання)

1. Що поверне `range(3, 10)` — скільки чисел і які саме?
2. Чому `while len(bombs) < BOMBS: ...` (Урок 6) не можна замінити на `for` без зміни логіки?
3. З Практикуму П1: якщо рішення A виконує `n` операцій, а рішення B — `n²`, у скільки разів B повільніше за A при `n = 1000`?

<details>
<summary>Відповіді</summary>

1. 7 чисел: `3, 4, 5, 6, 7, 8, 9`.
2. `for` проходить по колекції наперед відомої довжини за один прохід; тут наперед невідомо, скільки випадкових спроб знадобиться, щоб набрати саме `BOMBS` унікальних позицій — потрібна умова зупинки, а не фіксована кількість ітерацій.
3. У 1000 разів повільніше (`n² / n = n = 1000`).

</details>

## 📖 CONCEPT

### Від П1 до П2

Практикум П1 навчив рахувати ціну рішення. Природний наступний крок:

**«Можу просто перебрати все» → «А чи знаю я щось про дані, що дозволить зробити краще?»**

Відповідь залежить від того, що саме ми знаємо про дані:

```
UNSORTED         → linear scan          O(n)
SORTED           → binary search        O(log n)
                  | two pointers        O(n)
CONTIGUOUS RANGE  → sliding window       O(n)
```

Кожна властивість даних — окрема стратегія. Почнемо з найпростішої.

### 1. UNSORTED → лінійний пошук (коротко — ви це вже вмієте)

Якщо про порядок елементів нічого не відомо, єдиний надійний спосіб — перевірити кожен елемент по черзі, поки не знайдеться потрібний (або список не закінчиться):

In [1]:
def linear_search(data, target):
    for index, value in enumerate(data):
        if value == target:
            return index
    return -1


unsorted = [42, 7, 19, 3, 88, 15]

print(linear_search(unsorted, 19))   # знайдено -> індекс
print(linear_search(unsorted, 100))  # не знайдено -> -1

assert linear_search(unsorted, 19) == 2
assert linear_search(unsorted, 100) == -1
assert linear_search([], 5) == -1          # порожній список
assert linear_search([5], 5) == 0          # єдиний елемент
print("OK — лінійний пошук перевірено")

2
-1
OK — лінійний пошук перевірено


**O(n)** — у найгіршому випадку (елемента немає, або він останній) доведеться переглянути всі `n` елементів. Це базовий рівень, з яким порівнюємо все далі: якщо ми знаємо про дані більше, ніж «це просто список», можна зробити краще.

### 2. SORTED → бінарний пошук

Якщо дані **відсортовані**, не потрібно перевіряти кожен елемент: подивившись на середній елемент, одразу можна відкинути половину списку — вона свідомо не містить відповіді.

**PREDICT.** Список відсортований, `n = 1000` елементів. Скільки порівнянь щонайбільше знадобиться, щоб знайти (або не знайти) елемент? А якщо `n = 2000`? Запиши прогноз, перш ніж дивитись на код нижче.

In [2]:
def binary_search(data, target):
    """Повертає (індекс, кількість_порівнянь). Індекс -1, якщо не знайдено."""
    low, high = 0, len(data) - 1
    comparisons = 0
    while low <= high:
        mid = (low + high) // 2
        comparisons += 1
        if data[mid] == target:
            return mid, comparisons
        elif data[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1, comparisons


sorted_1000 = list(range(0, 2000, 2))   # 1000 відсортованих парних чисел
sorted_2000 = list(range(0, 4000, 2))   # 2000 відсортованих парних чисел

idx, cmp_1000 = binary_search(sorted_1000, 1998)   # останній елемент — найгірший випадок
_, cmp_2000 = binary_search(sorted_2000, 3998)

print(f"n=1000: індекс={idx}, порівнянь={cmp_1000}")
print(f"n=2000: порівнянь={cmp_2000}")

n=1000: індекс=999, порівнянь=10
n=2000: порівнянь=11


**INVESTIGATE.** Порівняй прогноз із реальним числом порівнянь надруку вище — `n = 1000` дало приблизно 10 порівнянь, `n = 2000` — приблизно 11: подвоєння `n` додало лише ОДНЕ порівняння. Це і є сенс **O(log n)**: кожне порівняння відкидає половину залишку, тож кількість кроків росте як `log₂(n)`, а не лінійно.

In [3]:
import math

assert cmp_1000 <= math.ceil(math.log2(1000)) + 1
assert cmp_2000 <= math.ceil(math.log2(2000)) + 1
assert cmp_2000 - cmp_1000 <= 2   # подвоєння n — щонайбільше +1..2 порівняння, НЕ вдвічі більше

# граничні випадки
assert binary_search([5], 5) == (0, 1)          # єдиний елемент
assert binary_search([5], 9)[0] == -1           # не знайдено
assert binary_search([1, 2, 3, 4, 5], 1)[0] == 0    # перший елемент
assert binary_search([1, 2, 3, 4, 5], 5)[0] == 4    # останній елемент
print("OK — бінарний пошук: прогноз підтверджено реальним підрахунком порівнянь")

OK — бінарний пошук: прогноз підтверджено реальним підрахунком порівнянь


### 3. SORTED → два вказівники (основна практика)

Класична задача **Two Sum**: знайти два числа у списку, сума яких дорівнює `target`. Список відсортований — і саме це дозволяє не перевіряти всі пари.

Ідея: один вказівник з початку (`left`), другий з кінця (`right`).
- Якщо сума замала — присортованому списку більший внесок дає лише зсув `left` праворуч (менші числа зліва).
- Якщо сума завелика — зсуваємо `right` ліворуч.
- Якщо сума дорівнює `target` — знайдено.

Кожен крок відкидає рівно одну можливість, а не одну пару — це і дає `O(n)` замість перевірки всіх пар.

In [4]:
def two_sum_sorted(numbers, target):
    """Повертає ((left, right), кількість_кроків) — індекси пари, або (None, кроки)."""
    left, right = 0, len(numbers) - 1
    steps = 0
    while left < right:
        steps += 1
        total = numbers[left] + numbers[right]
        if total == target:
            return (left, right), steps
        elif total < target:
            left += 1
        else:
            right -= 1
    return None, steps


numbers = [1, 2, 4, 7, 11, 15]
target = 9

pair, steps = two_sum_sorted(numbers, target)
left_idx, right_idx = pair
print(f"Індекси: {pair}, значення: {numbers[left_idx]} + {numbers[right_idx]} = {target}")
print(f"Кроків знадобилось: {steps}")

assert numbers[left_idx] + numbers[right_idx] == target
assert (numbers[left_idx], numbers[right_idx]) == (2, 7)

Індекси: (1, 3), значення: 2 + 7 = 9
Кроків знадобилось: 4


Порівняємо з наївним перебором усіх пар — той самий результат, але ціна інша:

In [5]:
def two_sum_bruteforce(numbers, target):
    checks = 0
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            checks += 1
            if numbers[i] + numbers[j] == target:
                return (i, j), checks
    return None, checks


brute_pair, brute_checks = two_sum_bruteforce(numbers, target)

print(f"Два вказівники: {steps} кроків")
print(f"Повний перебір:  {brute_checks} перевірок пар")

assert brute_pair == pair
assert steps < brute_checks
print("OK — той самий результат, менше кроків, бо скористались сортуванням")

Два вказівники: 4 кроків
Повний перебір:  7 перевірок пар
OK — той самий результат, менше кроків, бо скористались сортуванням


Граничні випадки — список без розв'язку, і список рівно з двох елементів:

In [6]:
assert two_sum_sorted([1, 2, 3], 100)[0] is None       # розв'язку немає
assert two_sum_sorted([2, 7], 9)[0] == (0, 1)          # мінімальний список — сама пара
print("OK — граничні випадки два вказівники перевірено")

OK — граничні випадки два вказівники перевірено


### 4. CONTIGUOUS RANGE → sliding window (друга основна практика)

Інша властивість даних: нас цікавить не будь-яка пара, а **суцільний відрізок** фіксованої довжини `k`. Задача: максимальна сума відрізка довжини `k` у списку чисел.

Спочатку — наївний спосіб: для кожної початкової позиції підсумовуємо `k` чисел заново:

In [7]:
def max_window_sum_naive(numbers, k):
    if k <= 0 or k > len(numbers):
        return None
    best = None
    for start in range(len(numbers) - k + 1):
        window_sum = sum(numbers[start:start + k])
        if best is None or window_sum > best:
            best = window_sum
    return best


data = [2, 1, 5, 1, 3, 2]
print(max_window_sum_naive(data, 3))   # найкраще вікно: [5, 1, 3] = 9

9


Наївний спосіб щоразу підсумовує `k` чисел заново — O(n·k). Але сусідні вікна відрізняються лише двома елементами: одне випадає зліва, одне додається справа. Не рахуємо суму заново — оновлюємо попередню:

In [8]:
def max_window_sum(numbers, k):
    if k <= 0 or k > len(numbers):
        return None
    window_sum = sum(numbers[:k])
    best = window_sum
    for i in range(k, len(numbers)):
        window_sum += numbers[i] - numbers[i - k]   # +новий елемент, -той що випав
        best = max(best, window_sum)
    return best


print(max_window_sum(data, 3))
assert max_window_sum(data, 3) == max_window_sum_naive(data, 3) == 9
print("OK — оптимізована версія дає той самий результат, що й наївна")

9
OK — оптимізована версія дає той самий результат, що й наївна


Граничні випадки: `k` більший за довжину списку, `k` дорівнює довжині списку, усі числа від'ємні (максимум все одно існує — просто «найменш від'ємне» вікно):

In [9]:
assert max_window_sum([1, 2, 3], 5) is None            # k > довжини списку
assert max_window_sum([1, 2, 3], 3) == 6                # k == довжині списку — одне вікно
assert max_window_sum([-5, -1, -8, -2], 2) == -6         # усі від'ємні: [-5, -1] -> -6, найкраще з гірших
print("OK — граничні випадки sliding window перевірено")

OK — граничні випадки sliding window перевірено


## 🔄 TRANSFER — та сама техніка, інша умова

Тепер — структурно ідентична задача, з іншою поверхневою деталлю: замість максимальної суми відрізка довжини `k` знайди **мінімальну**. Техніка (ковзне вікно, `+новий -старий`) та сама, лише порівняння `max`/`min` міняється місцями.

In [10]:
def min_window_sum(numbers, k):
    """Мінімальна сума суцільного відрізка довжини k. None, якщо k некоректне."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if k <= 0 or k > len(numbers):
        return None
    window_sum = sum(numbers[:k])
    best = window_sum
    for i in range(k, len(numbers)):
        window_sum += numbers[i] - numbers[i - k]
        best = min(best, window_sum)
    return best
    # END SOLUTION


data2 = [4, 2, 1, 7, 3, 6]

assert min_window_sum(data2, 2) == 3      # [2, 1] -> 3
assert min_window_sum(data2, 3) is not None
assert min_window_sum([1, 2, 3], 5) is None
print("OK — та сама техніка, інша умова порівняння")

OK — та сама техніка, інша умова порівняння


### Two Sum ще не закінчився

Ми щойно розв'язали Two Sum (`numbers = [1, 2, 4, 7, 11, 15]`, `target = 9`) двома вказівниками, бо список був **відсортований**. У Практикумі П3 (позиція 16) та сама задача, той самий `target = 9`, повернеться з **невідсортованими** даними — і тоді два вказівники напряму вже не спрацюють: сортувати заново — це вже своя ціна, а `dict`/`set`-підхід дає інший шлях до розв'язку. Той самий принцип, що вже звучав у П1→П2: **алгоритм залежить не тільки від задачі, а й від представлення даних.**

## ✅ Самоперевірка (5 запитань)

**1.** Чому лінійний пошук не можна прискорити до `O(log n)`, якщо дані невідсортовані?

<details><summary>Відповідь</summary>Бінарний пошук відкидає половину даних на кожному кроці, спираючись на порядок (якщо середній елемент менший за ціль — ціль точно не ліворуч). Без порядку такого висновку зробити не можна — доведеться перевіряти елементи по черзі, тобто <code>O(n)</code>.</details>

**2.** Два вказівники для Two Sum вимагають відсортованих даних. Що станеться, якщо застосувати цю ж техніку до невідсортованого списку?

<details><summary>Відповідь</summary>Висновки «сума замала — зсунь <code>left</code>» / «сума завелика — зсунь <code>right</code>» спираються на те, що зсув вказівника монотонно змінює суму — це правда лише для відсортованих даних. На невідсортованому списку зсув вказівника може дати будь-яку зміну суми, і алгоритм більше не гарантує знаходження пари.</details>

**3.** У наївній версії sliding window складність `O(n·k)`. Чому оптимізована версія — `O(n)`?

<details><summary>Відповідь</summary>Наївна версія щоразу підсумовує всі <code>k</code> елементів вікна заново. Оптимізована переиспользує попередню суму: віднімає елемент, що випав зліва, додає новий справа — це стала (<code>O(1)</code>) робота на кожен крок, а кроків <code>n - k + 1</code>, тобто разом <code>O(n)</code>.</details>

**4.** Чому бінарний пошук на `n = 2000` елементів потребує лише на 1 порівняння більше, ніж на `n = 1000`?

<details><summary>Відповідь</summary>Кожне порівняння відкидає половину залишку. Кількість кроків росте як <code>log₂(n)</code>: <code>log₂(2000) - log₂(1000) = log₂(2) = 1</code> — подвоєння даних додає рівно один додатковий крок, а не вдвічі більше кроків.</details>

**5.** Дано відсортований список і задачу Two Sum. Чому два вказівники (`O(n)`) кращі за бінарний пошук для кожного елемента (`O(n log n)`)?

<details><summary>Відповідь</summary>Бінарний пошук «для кожного числа шукати доповнення» справді працює (<code>O(n log n)</code>), але два вказівники розв'язують всю задачу за один прохід (<code>O(n)</code>), бо не перезапускають пошук з нуля для кожного елемента — вони використовують результат попереднього кроку (звуження меж).</details>

### Шпаргалка

| Властивість даних | Стратегія | Складність |
|---|---|---|
| Немає нічого відомого | лінійний пошук | O(n) |
| Відсортовані | бінарний пошук (пошук одного значення) | O(log n) |
| Відсортовані | два вказівники (пара з певною сумою) | O(n) |
| Суцільний відрізок фіксованої довжини | sliding window | O(n) |

```python
# бінарний пошук — скорочуємо межі
while low <= high:
    mid = (low + high) // 2
    ...

# два вказівники — звужуємо з обох кінців
while left < right:
    total = numbers[left] + numbers[right]
    ...

# sliding window — переиспользуємо суму
window_sum += numbers[i] - numbers[i - k]
```

## Далі

Практикум П3 (позиція 16, «Хеш-структури») повертається до Two Sum з тими самими числами й тим самим `target = 9`, але вже без сортування — там `dict`/`set` дадуть інший шлях до відповіді, коли два вказівники більше не застосовні напряму.